## 0. Installation and imports

In [1]:
import subprocess, sys
try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=0.9.12'])

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Ambiente: {"Google Colab" if IN_COLAB else "Local"}')

In [3]:
import json
import random
import time
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm

from PIL import Image
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              average_precision_score, roc_auc_score)

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'timm    : {timm.__version__}')

## 1. Configuration

In [4]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/anonymous')
    IMGS_DIR = Path('/content/Imgs')
    import os
    if not os.path.exists('/content/Imgs'):
        print('Descompactando imagens para o disco rápido do Colab...')
        !unzip -q /content/drive/MyDrive/anonymous/Data/Imgs.zip -d /content/
        print('Descompactação concluída!')
else:
    BASE_DIR = Path('e:/Endo-ICTAI-2026')
    IMGS_DIR = Path('e:/Endo-ICTAI-2026/Data/Imgs')

SPLITS_DIR  = BASE_DIR / 'splits'
RESULTS_DIR = BASE_DIR / 'results' / 'nb04'
CKPT_DIR    = BASE_DIR / 'checkpoints'
FIGS_DIR    = BASE_DIR / 'figs' / 'training'

for d in [RESULTS_DIR, CKPT_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CORE_LABELS = ['ENANTEMA', 'PÓLIPO', 'ÚLCERA', 'EROSÃO', 'MICRONODULARIDADE']
N_FOLDS     = 5
N_CLASSES   = len(CORE_LABELS)

IMG_SIZE     = 224
BATCH_SIZE   = 128
MAX_EPOCHS   = 50
LR_BACKBONE  = 1e-4
LR_HEAD      = 1e-3
WEIGHT_DECAY = 1e-4
ES_PATIENCE  = 10
LR_PATIENCE  = 4
LR_FACTOR    = 0.5
SEEDS        = [42, 43, 44, 45, 46]
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

EXPERIMENTS = [
    {'id': 'wrs_bce',      'loss': 'bce',      'sampler': 'wrs',     'lam': 0.0},
    {'id': 'focal_g2',     'loss': 'focal',    'sampler': 'uniform', 'lam': 0.0},
    {'id': 'focal_wrs',    'loss': 'focal',    'sampler': 'wrs',     'lam': 0.0},
    {'id': 'wrs_m2_l06',   'loss': 'm2',       'sampler': 'wrs',     'lam': 0.6},
    {'id': 'focal_m2_l06', 'loss': 'focal_m2', 'sampler': 'wrs',     'lam': 0.6},
    {'id': 'asl',          'loss': 'asl',      'sampler': 'uniform', 'lam': 0.0},
    {'id': 'asl_m2_l06',   'loss': 'asl_m2',   'sampler': 'uniform', 'lam': 0.6},
]

print(f'Device      : {DEVICE}')
print(f'Experimentos: {len(EXPERIMENTS)} × {N_FOLDS} folds = {len(EXPERIMENTS) * N_FOLDS} runs')
print(f'Tempo estimado: ~{len(EXPERIMENTS) * N_FOLDS * 10 / 60:.0f}h')

## 2. Dataset and transforms (AUG-02, same as NB03)

In [5]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_transforms(split: str) -> T.Compose:
    normalize = T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    if split == 'train':
        return T.Compose([
            T.Resize((int(IMG_SIZE * 1.15), int(IMG_SIZE * 1.15))),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=10),
            T.RandomResizedCrop(size=IMG_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
            T.ToTensor(),
            normalize,
        ])
    else:
        return T.Compose([
            T.Resize((int(IMG_SIZE * 1.14), int(IMG_SIZE * 1.14))),
            T.CenterCrop(IMG_SIZE),
            T.ToTensor(),
            normalize,
        ])

class GastroscopyDataset(Dataset):
    def __init__(self, df, imgs_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.imgs_dir  = imgs_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        img    = Image.open(self.imgs_dir / row['image_name']).convert('RGB')
        labels = torch.tensor(row[CORE_LABELS].values.astype(float), dtype=torch.float32)
        if self.transform:
            img = self.transform(img)
        return img, labels

def compute_pos_weights(df) -> torch.Tensor:
    pos = df[CORE_LABELS].sum(axis=0).values
    neg = len(df) - pos
    return torch.tensor(neg / (pos + 1e-6), dtype=torch.float32).to(DEVICE)

def compute_cooccurrence_matrix(df) -> torch.Tensor:
    labels = df[CORE_LABELS].values.astype(float)
    C = np.zeros((N_CLASSES, N_CLASSES))
    for i in range(N_CLASSES):
        mask_i = labels[:, i] == 1
        n_i    = mask_i.sum()
        if n_i == 0:
            continue
        for j in range(N_CLASSES):
            if i == j:
                continue
            C[i][j] = labels[mask_i, j].sum() / n_i
    return torch.tensor(C, dtype=torch.float32).to(DEVICE)

print('Dataset e funções auxiliares definidos.')

## 3. Loss Functions: BCE, Focal Loss (γ=2), M2 Co-occurrence

In [6]:
class MultilabelFocalLoss(nn.Module):
    """
    Binary Focal Loss adaptada para multilabel com K classes independentes.

    FL(p_t) = -(1 - p_t)^gamma * log(p_t)

    pos_weight: amplifica o gradiente das classes raras (mesmo esquema da BCE).
    gamma=2: recomendado para desbalanceamento severo; gamma=0 reduz a BCE padrão.
    """
    def __init__(self, gamma: float = 2.0, pos_weight: torch.Tensor = None):
        super().__init__()
        self.gamma      = gamma
        self.pos_weight = pos_weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce = F.binary_cross_entropy_with_logits(
            logits, targets,
            pos_weight=self.pos_weight,
            reduction='none'
        )
        probs   = torch.sigmoid(logits)
        pt      = targets * probs + (1 - targets) * (1 - probs)
        focal_w = (1 - pt).pow(self.gamma)
        return (focal_w * bce).mean()

class AsymmetricLoss(nn.Module):
    """
    Asymmetric Loss (ASL) — Ben-Baruch et al., NeurIPS 2021.

    Positivos: gamma_pos=0 → BCE pura (sem focusing), estável via logsigmoid.
    Negativos: probability margin (clip) + gamma_neg alto downweightam negativos
               fáceis que dominam o gradiente no desbalanceamento severo (IR>>1).

    Configuração recomendada para multilabel severo:
      gamma_neg=4, gamma_pos=0, clip=0.05
    """
    def __init__(self, gamma_neg: float = 4, gamma_pos: float = 0,
                 clip: float = 0.05, pos_weight: torch.Tensor = None):
        super().__init__()
        self.gamma_neg  = gamma_neg
        self.gamma_pos  = gamma_pos
        self.clip       = clip
        self.pos_weight = pos_weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(logits)

        log_pos = targets * F.logsigmoid(logits)
        if self.pos_weight is not None:
            log_pos = log_pos * self.pos_weight

        xs_neg = (1 - probs) * (1 - targets)
        if self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)
        log_neg = (1 - targets) * torch.log(xs_neg.clamp(min=1e-8))

        loss = log_pos + log_neg

        pt    = probs * targets + xs_neg * (1 - targets)
        gamma = self.gamma_pos * targets + self.gamma_neg * (1 - targets)
        loss *= (1 - pt).pow(gamma)

        return -loss.mean()

class M2CoocLoss(nn.Module):
    """
    L_M2 = base_loss(logits, targets) + λ * L_coo

    base_loss: qualquer loss binária (BCE ou Focal ou ASL) — injetada no construtor.
    Isso permite combinar M2 com Focal Loss (focal_m2_l06), BCE (wrs_m2_l06)
    ou ASL (asl_m2_l06).

    L_coo penaliza inconsistência entre predições e co-ocorrências clínicas do treino:
        L_coo = Σ_{i≠j} C[i][j] * relu(p_i - p_j)²
    C computado por fold a partir do treino — sem leakage.
    """
    def __init__(self, base_loss: nn.Module, cooc_matrix: torch.Tensor, lam: float):
        super().__init__()
        self.base_loss = base_loss
        self.C         = cooc_matrix
        self.lam       = lam

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_base = self.base_loss(logits, targets)
        if self.lam == 0.0:
            return loss_base
        probs    = torch.sigmoid(logits)
        p_i      = probs.unsqueeze(2)
        p_j      = probs.unsqueeze(1)
        diff     = torch.relu(p_i - p_j)
        loss_coo = (self.C.unsqueeze(0) * diff.pow(2)).mean()
        return loss_base + self.lam * loss_coo

def build_criterion(loss_type: str, pos_weight: torch.Tensor,
                    cooc_matrix: torch.Tensor = None, lam: float = 0.0):
    """
    Factory de losses. Aceita:
      'bce'       → BCEWithLogitsLoss + pos_weight
      'focal'     → MultilabelFocalLoss(γ=2) + pos_weight
      'asl'       → AsymmetricLoss(γ_neg=4, γ_pos=0, clip=0.05) + pos_weight
      'asl_m2'    → M2CoocLoss com base ASL + pos_weight  (combinação nova)
      'm2'        → M2CoocLoss com base BCE + pos_weight
      'focal_m2'  → M2CoocLoss com base Focal(γ=2) + pos_weight
    """
    if loss_type == 'bce':
        return nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    elif loss_type == 'focal':
        return MultilabelFocalLoss(gamma=2.0, pos_weight=pos_weight)
    elif loss_type == 'asl':
        return AsymmetricLoss(gamma_neg=4, gamma_pos=0, clip=0.05,
                              pos_weight=pos_weight)
    elif loss_type == 'asl_m2':
        base = AsymmetricLoss(gamma_neg=4, gamma_pos=0, clip=0.05,
                              pos_weight=pos_weight)
        return M2CoocLoss(base, cooc_matrix, lam)
    elif loss_type == 'm2':
        base = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        return M2CoocLoss(base, cooc_matrix, lam)
    elif loss_type == 'focal_m2':
        base = MultilabelFocalLoss(gamma=2.0, pos_weight=pos_weight)
        return M2CoocLoss(base, cooc_matrix, lam)
    else:
        raise ValueError(f'Loss desconhecida: {loss_type}')

print('Loss functions definidas: BCE, Focal(γ=2), ASL(γ_neg=4, clip=0.05), ASL+M2, M2+BCE, M2+Focal.')

## 4. WeightedRandomSampler (WRS) multilabel

In [7]:
def build_multilabel_sampler(df) -> WeightedRandomSampler:
    """
    Peso de cada amostra = inverso da frequência da sua classe mais rara presente.
    Amostras sem rótulo ativo recebem peso mínimo (1.0).
    Garante que batches contenham mais exemplos de PÓLIPO e MICRONODULARIDADE.
    """
    labels_matrix = df[CORE_LABELS].values
    freq     = labels_matrix.sum(axis=0) / len(labels_matrix)
    inv_freq = 1.0 / (freq + 1e-6)

    sample_weights = []
    for row in labels_matrix:
        active = np.where(row == 1)[0]
        if len(active) == 0:
            sample_weights.append(1.0)
        else:
            sample_weights.append(float(inv_freq[active].max()))

    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

df_tmp = pd.read_csv(SPLITS_DIR / 'fold_0_train.csv')
sampler_tmp = build_multilabel_sampler(df_tmp)
w = np.array(sampler_tmp.weights)
print('WRS — peso médio por classe (fold 0):')
for i, label in enumerate(CORE_LABELS):
    mask = df_tmp[label].values == 1
    print(f'  {label:<20}: n={mask.sum():4d}  peso_médio={w[mask].mean():.1f}')
print(f'  sem rótulo ativo   : n={(df_tmp[CORE_LABELS].sum(axis=1)==0).sum():4d}  peso=1.0')

## 5. Model (Swin-Tiny) and optimizer

In [8]:
def build_swin_tiny(n_classes: int = N_CLASSES) -> nn.Module:
    return timm.create_model(
        'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k',
        pretrained=True,
        num_classes=n_classes
    )

def build_optimizer(model: nn.Module):
    head_params     = list(model.head.parameters())
    backbone_params = [p for n, p in model.named_parameters() if 'head' not in n]
    return torch.optim.AdamW(
        [
            {'params': backbone_params, 'lr': LR_BACKBONE},
            {'params': head_params,     'lr': LR_HEAD},
        ],
        weight_decay=WEIGHT_DECAY,
    )

print('Modelo Swin-Tiny e otimizador definidos.')

## 6. Metrics

In [9]:
def optimize_thresholds(probs: np.ndarray, labels: np.ndarray) -> np.ndarray:
    thresholds = np.zeros(N_CLASSES)
    for i in range(N_CLASSES):
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.1, 0.9, 0.05):
            preds = (probs[:, i] >= t).astype(int)
            if preds.sum() == 0:
                continue
            f1 = f1_score(labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[i] = best_t
    return thresholds

def compute_metrics(probs: np.ndarray, labels: np.ndarray,
                    thresholds: np.ndarray = None) -> dict:
    if thresholds is None:
        thresholds = np.full(N_CLASSES, 0.5)
    preds = (probs >= thresholds).astype(int)

    f1_per = f1_score(labels, preds, average=None, zero_division=0)
    pr_per = precision_score(labels, preds, average=None, zero_division=0)
    rc_per = recall_score(labels, preds, average=None, zero_division=0)

    auprc, rocauc = [], []
    for i in range(N_CLASSES):
        auprc.append(average_precision_score(labels[:, i], probs[:, i])
                     if labels[:, i].sum() > 0 else float('nan'))
        rocauc.append(roc_auc_score(labels[:, i], probs[:, i])
                      if 0 < labels[:, i].sum() < len(labels) else float('nan'))

    multi_mask = labels.sum(axis=1) >= 2
    f1_multi   = float(f1_score(labels[multi_mask], preds[multi_mask],
                                 average='macro', zero_division=0))\
                 if multi_mask.sum() > 0 else float('nan')

    result = {
        'f1_macro':      float(np.nanmean(f1_per)),
        'f1_micro':      float(f1_score(labels, preds, average='micro', zero_division=0)),
        'hamming':       float(np.mean(preds != labels)),
        'f1_multi':      f1_multi,
        'pr_auc_macro':  float(np.nanmean(auprc)),
        'roc_auc_macro': float(np.nanmean(rocauc)),
    }
    for i, label in enumerate(CORE_LABELS):
        result[f'f1_{label}']     = float(f1_per[i])
        result[f'prec_{label}']   = float(pr_per[i])
        result[f'rec_{label}']    = float(rc_per[i])
        result[f'auprc_{label}']  = float(auprc[i])
        result[f'rocauc_{label}'] = float(rocauc[i])
        result[f'thr_{label}']    = float(thresholds[i])
    return result

print('Funções de métricas definidas.')

## 7. Training loop

In [10]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = True

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        logits = model(imgs.to(DEVICE))
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)

def train_one_fold(fold: int, exp_config: dict,
                   save_checkpoint: bool = True,
                   verbose: bool = True) -> dict:
    """
    Treina Swin-Tiny em um fold com a configuração dada.
    exp_config: {'id', 'loss', 'sampler', 'lam'}
    """
    exp_id = exp_config['id']
    set_seed(SEEDS[fold])

    df_tr  = pd.read_csv(SPLITS_DIR / f'fold_{fold}_train.csv')
    df_val = pd.read_csv(SPLITS_DIR / f'fold_{fold}_val.csv')
    df_te  = pd.read_csv(SPLITS_DIR / f'fold_{fold}_test.csv')

    ds_tr  = GastroscopyDataset(df_tr,  IMGS_DIR, get_transforms('train'))
    ds_val = GastroscopyDataset(df_val, IMGS_DIR, get_transforms('val'))
    ds_te  = GastroscopyDataset(df_te,  IMGS_DIR, get_transforms('val'))

    if exp_config['sampler'] == 'wrs':
        sampler = build_multilabel_sampler(df_tr)
        dl_tr   = DataLoader(ds_tr, batch_size=BATCH_SIZE, sampler=sampler,
                             num_workers=0, pin_memory=True)
    else:
        dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=0, pin_memory=True)

    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True)
    dl_te  = DataLoader(ds_te,  batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True)

    model    = build_swin_tiny().to(DEVICE)
    pos_w    = compute_pos_weights(df_tr)

    needs_cooc = exp_config['loss'] in ('m2', 'focal_m2', 'asl_m2')
    cooc_mat   = compute_cooccurrence_matrix(df_tr) if needs_cooc else None
    criterion  = build_criterion(exp_config['loss'], pos_w, cooc_mat, exp_config['lam'])
    optimizer  = build_optimizer(model)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=LR_PATIENCE, factor=LR_FACTOR
    )

    best_val_f1  = -1.0
    best_weights = None
    es_counter   = 0
    history      = {'train_loss': [], 'val_f1': []}

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        for imgs, labels in dl_tr:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(imgs)

        train_loss = running_loss / len(ds_tr)

        val_probs, val_labels = evaluate(model, dl_val)
        val_thr = optimize_thresholds(val_probs, val_labels)
        val_f1  = compute_metrics(val_probs, val_labels, val_thr)['f1_macro']

        scheduler.step(val_f1)
        history['train_loss'].append(train_loss)
        history['val_f1'].append(val_f1)

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f'  Epoch {epoch:>3} | loss={train_loss:.4f} | val_F1={val_f1:.4f}')

        if val_f1 > best_val_f1 + 1e-4:
            best_val_f1  = val_f1
            best_weights = deepcopy(model.state_dict())
            es_counter   = 0
        else:
            es_counter += 1
            if es_counter >= ES_PATIENCE:
                if verbose:
                    print(f'  Early stopping na época {epoch}.')
                break

    model.load_state_dict(best_weights)
    if save_checkpoint:
        torch.save(best_weights, CKPT_DIR / f'swin_nb04_{exp_id}_fold{fold}.pt')

    test_probs, test_labels = evaluate(model, dl_te)
    test_met = compute_metrics(test_probs, test_labels, val_thr)
    test_met.update({
        'fold': fold, 'exp_id': exp_id,
        'best_val_f1': best_val_f1,
        'epochs_trained': len(history['train_loss']),
        'val_thresholds': val_thr.tolist(),
        'history': history,
        'test_preds': (test_probs >= val_thr).astype(int).tolist(),
        'test_labels': test_labels.tolist(),
    })
    return test_met

print('Loop de treino definido.')

## 8. Execution of experiments

In [11]:
RUN_FAST_CHECK = False

if RUN_FAST_CHECK:
    MAX_EPOCHS   = 5
    FOLDS_TO_RUN = [0]
    print('MODO FAST CHECK: 1 fold, 5 épocas.')
else:
    FOLDS_TO_RUN = list(range(N_FOLDS))
    print(f'MODO COMPLETO: {len(EXPERIMENTS)} experimentos × {N_FOLDS} folds = '
          f'{len(EXPERIMENTS) * N_FOLDS} runs.')
    print(f'Tempo estimado: ~{len(EXPERIMENTS) * N_FOLDS * 10 / 60:.0f}h')

In [12]:
results_file = RESULTS_DIR / 'nb04_results.json'

if results_file.exists():
    print(">> ENCONTRADO RESULTADO ANTERIOR! Retomando progresso... <<")
    with open(results_file, 'r', encoding='utf-8') as f:
        all_results = json.load(f)
else:
    all_results = {}

for exp in EXPERIMENTS:
    exp_id = exp['id']

    if exp_id in all_results and len(all_results[exp_id].get('fold_results', [])) >= len(FOLDS_TO_RUN):
        print(f'\n[PULANDO] Experimento {exp_id} já concluído anteriormente.')
        continue

    print(f'\n{"="*60}')
    print(f'Experimento: {exp_id}')
    print(f'  loss={exp["loss"]}  sampler={exp["sampler"]}  λ={exp["lam"]}')
    print(f'{"="*60}')

    fold_results = []
    t0 = time.time()

    for fold in FOLDS_TO_RUN:
        print(f'\n  Fold {fold}/{N_FOLDS-1}')
        met = train_one_fold(fold, exp, save_checkpoint=not RUN_FAST_CHECK)
        fold_results.append(met)
        print(f'  → Test Macro-F1={met["f1_macro"]:.4f}  '
              f'PÓLIPO={met["f1_PÓLIPO"]:.4f}  '
              f'MICRONOD={met["f1_MICRONODULARIDADE"]:.4f}')

    elapsed = time.time() - t0

    metric_keys = (['f1_macro', 'f1_micro', 'hamming', 'f1_multi',
                    'pr_auc_macro', 'roc_auc_macro'] +
                   [f'f1_{l}' for l in CORE_LABELS] +
                   [f'auprc_{l}' for l in CORE_LABELS])

    agg = {}
    for k in metric_keys:
        vals = [r[k] for r in fold_results
                if not np.isnan(r.get(k, float('nan')))]
        if vals:
            agg[k] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals))}

    all_results[exp_id] = {
        'config':       exp,
        'fold_results': fold_results,
        'aggregated':   agg,
        'elapsed_sec':  round(elapsed, 1),
    }

    print(f'\n  AGREGADO: Macro-F1 = '
          f'{agg["f1_macro"]["mean"]:.4f} ± {agg["f1_macro"]["std"]:.4f}')
    print(f'  Tempo: {elapsed/60:.1f} min')

    with open(results_file, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print('  Salvo parcialmente.')

## 9. Statistical analysis — Bootstrap vs NB03 M0 Swin-Tiny

In [13]:
NB03_RESULTS = BASE_DIR / 'results' / 'm2' / 'm2_results.json'

with open(NB03_RESULTS, encoding='utf-8') as f:
    nb03_data = json.load(f)

def collect_preds_labels_nb03(exp_id):
    all_p, all_l = [], []
    for r in nb03_data[exp_id]['fold_results']:
        all_p.extend(r['test_preds'])
        all_l.extend(r['test_labels'])
    return np.array(all_p), np.array(all_l)

def collect_preds_labels(exp_id):
    all_p, all_l = [], []
    for r in all_results[exp_id]['fold_results']:
        all_p.extend(r['test_preds'])
        all_l.extend(r['test_labels'])
    return np.array(all_p), np.array(all_l)

def bootstrap_compare(preds_a, labels_a, preds_b, labels_b, B=2000, seed=42):
    rng = np.random.default_rng(seed)
    n   = len(labels_a)
    deltas = []
    for _ in range(B):
        idx  = rng.integers(0, n, size=n)
        f1_a = f1_score(labels_a[idx], preds_a[idx], average='macro', zero_division=0)
        f1_b = f1_score(labels_b[idx], preds_b[idx], average='macro', zero_division=0)
        deltas.append(f1_b - f1_a)
    deltas = np.array(deltas)
    return {
        'delta_mean':  float(deltas.mean()),
        'ci_95_low':   float(np.percentile(deltas, 2.5)),
        'ci_95_high':  float(np.percentile(deltas, 97.5)),
        'p_value':     float(np.mean(deltas <= 0)),
        'significant': bool(np.percentile(deltas, 2.5) > 0),
    }

preds_base, labels_base = collect_preds_labels_nb03('m0_swin')
base_f1 = f1_score(labels_base, preds_base, average='macro', zero_division=0)
print(f'Referência — M0 Swin-Tiny (NB03): F1={base_f1:.4f}')
print()

comparisons = {}
print(f'{"Experimento":<14} {"F1-macro":>10} {"Δ vs M0":>8} {"p-value":>10} {"Sig.":>6}')
print('-' * 55)

for exp_id in all_results:
    preds_exp, labels_exp = collect_preds_labels(exp_id)
    exp_f1 = f1_score(labels_exp, preds_exp, average='macro', zero_division=0)
    boot   = bootstrap_compare(preds_base, labels_base, preds_exp, labels_exp)
    sig    = '✓' if boot['significant'] else '—'
    print(f'{exp_id:<14} {exp_f1:>10.4f} {boot["delta_mean"]:>+8.4f} '
          f'{boot["p_value"]:>10.4f} {sig:>6}')
    comparisons[exp_id] = {
        'f1_agrupado': float(exp_f1), **boot
    }

with open(RESULTS_DIR / 'bootstrap_comparisons.json', 'w', encoding='utf-8') as f:
    json.dump(comparisons, f, ensure_ascii=False, indent=2)
print('\nComparações salvas.')

## 10. Figures

In [14]:
m0_agg = nb03_data['m0_swin']['aggregated']

exp_labels = ['M0\n(ref NB03)'] + [e['id'].replace('_', '\n') for e in EXPERIMENTS]

f1_means = [m0_agg['f1_macro']['mean']] + \
           [all_results[e['id']]['aggregated']['f1_macro']['mean'] for e in EXPERIMENTS]
f1_stds  = [m0_agg['f1_macro']['std']] + \
           [all_results[e['id']]['aggregated']['f1_macro']['std'] for e in EXPERIMENTS]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]

colors = ['#90A4AE', '#1565C0', '#E65100', '#AD1457', '#2E7D32', '#6A1B9A']
bars = ax.bar(exp_labels, f1_means, yerr=f1_stds, color=colors,
              capsize=5, edgecolor='white', linewidth=0.5)
ax.axhline(f1_means[0], color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax.set_ylabel('Macro-F1 (média ± std, 5 folds)', fontsize=10)
ax.set_title('NB04 — WRS + Focal vs Referência M0', fontweight='bold')
ax.set_ylim(0.45, min(1.0, max(f1_means) + 0.12))
for bar, mean in zip(bars, f1_means):
    ax.text(bar.get_x() + bar.get_width()/2, mean + 0.01,
            f'{mean:.3f}', ha='center', va='bottom', fontsize=8.5)
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
heat_data = {}

heat_data['M0 (ref)'] = [m0_agg.get(f'f1_{l}', {}).get('mean', float('nan'))
                          for l in CORE_LABELS]
for exp_id, res in all_results.items():
    heat_data[exp_id] = [res['aggregated'].get(f'f1_{l}', {}).get('mean', float('nan'))
                         for l in CORE_LABELS]

df_heat = pd.DataFrame(heat_data, index=CORE_LABELS).T
sns.heatmap(df_heat, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0.3, vmax=0.85, linewidths=0.4, ax=ax,
            cbar_kws={'label': 'F1'})
ax.set_title('F1 por classe × experimento', fontweight='bold')

plt.suptitle('NB04 — Estratégias de Desbalanceamento (Swin-Tiny)', fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(FIGS_DIR / 'nb04_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 11. Comparative table and final summary

In [15]:
rows = []

REF_TRILHA3_M0 = {'f1_macro': 0.517, 'f1_ENANTEMA': 0.538, 'f1_PÓLIPO': 0.507,
                   'f1_ÚLCERA': 0.520, 'f1_EROSÃO': 0.670, 'f1_MICRONODULARIDADE': 0.351}
REF_TRILHA3_M2 = {'f1_macro': 0.537, 'f1_ENANTEMA': 0.470, 'f1_PÓLIPO': 0.624,
                   'f1_ÚLCERA': 0.520, 'f1_EROSÃO': 0.628, 'f1_MICRONODULARIDADE': 0.409}

for label, ref in [('Trilha-3 M0 (ResNet50)', REF_TRILHA3_M0),
                    ('Trilha-3 M2 λ=0.3 (ResNet50)', REF_TRILHA3_M2)]:
    row = {'Modelo': label}
    for k in ['f1_macro', 'f1_ENANTEMA', 'f1_PÓLIPO',
               'f1_ÚLCERA', 'f1_EROSÃO', 'f1_MICRONODULARIDADE']:
        row[k.replace('f1_', '')] = f"{ref[k]:.3f}"
    rows.append(row)

row = {'Modelo': 'M0 Swin-Tiny (NB03 ref)'}
for k in ['f1_macro', 'f1_ENANTEMA', 'f1_PÓLIPO',
           'f1_ÚLCERA', 'f1_EROSÃO', 'f1_MICRONODULARIDADE']:
    v = m0_agg.get(k)
    row[k.replace('f1_', '')] = f"{v['mean']:.3f}±{v['std']:.3f}" if v else '-'
rows.append(row)

for exp_id, res in all_results.items():
    agg = res['aggregated']
    cfg = res['config']
    label = f"NB04 {exp_id} (loss={cfg['loss']}, sampler={cfg['sampler']})"
    row = {'Modelo': label}
    for k in ['f1_macro', 'f1_ENANTEMA', 'f1_PÓLIPO',
               'f1_ÚLCERA', 'f1_EROSÃO', 'f1_MICRONODULARIDADE']:
        v = agg.get(k)
        row[k.replace('f1_', '')] = f"{v['mean']:.3f}±{v['std']:.3f}" if v else '-'
    rows.append(row)

df_table = pd.DataFrame(rows).set_index('Modelo')
print('=== TABELA COMPARATIVA COMPLETA ===')
print(df_table.to_string())
df_table.to_csv(RESULTS_DIR / 'tabela_nb04.csv')

print('\n=== RESUMO FINAL ===')
best_exp = max(
    [(e, r['aggregated']['f1_macro']['mean']) for e, r in all_results.items()],
    key=lambda x: x[1]
)
m0_mean = m0_agg['f1_macro']['mean']
print(f'M0 Swin-Tiny (NB03)   : {m0_mean:.4f}')
print(f'Melhor NB04           : {best_exp[0]} (F1={best_exp[1]:.4f})')
print(f'Δ vs M0               : {best_exp[1] - m0_mean:+.4f}')

print('\nF1 PÓLIPO por experimento:')
m0_polipo = m0_agg['f1_PÓLIPO']['mean']
print(f'  M0 Swin-Tiny (ref)  : {m0_polipo:.4f}')
for exp_id, res in all_results.items():
    v = res['aggregated'].get('f1_PÓLIPO', {}).get('mean', float('nan'))
    print(f'  {exp_id:<20}: {v:.4f}  ({v - m0_polipo:+.4f})')

print('\nF1 MICRONODULARIDADE por experimento:')
m0_micro = m0_agg['f1_MICRONODULARIDADE']['mean']
print(f'  M0 Swin-Tiny (ref)  : {m0_micro:.4f}')
for exp_id, res in all_results.items():
    v = res['aggregated'].get('f1_MICRONODULARIDADE', {}).get('mean', float('nan'))
    print(f'  {exp_id:<20}: {v:.4f}  ({v - m0_micro:+.4f})')

In [16]:
import time
print('Treinamento finalizado. A sessão será encerrada em 5 minutos para economizar créditos do Colab.')
time.sleep(300)
if IN_COLAB:
    from google.colab import runtime
    runtime.unassign()